In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df=pd.read_parquet("/Users/anirudhgoyal/Desktop/Projects/lob-midprice-prediction/data/aapl_stage2.parquet")
print(f"Loaded: {df.shape}")
df.head()


Loaded: (400391, 48)


,time_sec,time_hour,ask_price_1,ask_size_1,bid_price_1,bid_size_1,mid_price,spread,ofi_top,ofi_deep,...,bid_price_10,bid_size_2,bid_size_3,bid_size_4,bid_size_5,bid_size_6,bid_size_7,bid_size_8,bid_size_9,bid_size_10
0,34200.004241,9.500001,585.94,200,585.33,18,585.635,0.61,-0.834862,-0.256965,...,584.27,150,5,89,5,300,300,300,200,300
1,34200.004261,9.500001,585.94,200,585.33,18,585.635,0.61,-0.834862,-0.341260,...,584.38,18,150,5,89,5,300,300,300,200
2,34200.004447,9.500001,585.94,200,585.33,18,585.635,0.61,-0.834862,-0.401939,...,584.53,18,18,150,5,89,5,300,300,300
3,34200.025552,9.500007,585.91,18,585.33,18,585.620,0.58,0.000000,-0.320531,...,584.53,18,18,150,5,89,5,300,300,300
4,34200.025580,9.500007,585.91,18,585.33,18,585.620,0.58,0.000000,0.002918,...,584.53,18,18,150,5,89,5,300,300,300


In [22]:
df["mid_price"] = (df["ask_price_1"] + df["bid_price_1"]) / 2
df["spread"] = df["ask_price_1"] - df["bid_price_1"]
df["log_spread"] = np.log(df["spread"] + 1e-9)  # +tiny number to avoid log(0)


In [23]:

# Volume at top of book
df["bid_volume_top"] = df["bid_size_1"]
df["ask_volume_top"] = df["ask_size_1"]
df["total_volume_top"] = df["bid_size_1"] + df["ask_size_1"]

In [24]:
print(df[["spread", "log_spread"]].describe())

              spread     log_spread
count  400391.000000  400391.000000
mean        0.153489      -2.011395
std         0.074158       0.577128
min         0.010000      -4.605170
25%         0.100000      -2.302585
50%         0.150000      -1.897120
75%         0.200000      -1.609438
max         0.920000      -0.083382


In [25]:
# Weighted OFI — closer levels matter more
# Weight = 1, 0.5, 0.33, 0.25, 0.2, 0.17, 0.14, 0.125, 0.11, 0.1 (= 1/level)
weights = np.array([1/i for i in range(1, 11)])

bid_size_cols = [f"bid_size_{i}" for i in range(1, 11)]
ask_size_cols = [f"ask_size_{i}" for i in range(1, 11)]

# Weighted sums
weighted_bid = (df[bid_size_cols] * weights).sum(axis=1)
weighted_ask = (df[ask_size_cols] * weights).sum(axis=1)

df["ofi_weighted"] = (weighted_bid - weighted_ask) / (weighted_bid + weighted_ask)

# Also: just level 1, 3, 5 OFI for different "depths" of pressure
for level in [1, 3, 5]:
    bid_sum = df[[f"bid_size_{i}" for i in range(1, level+1)]].sum(axis=1)
    ask_sum = df[[f"ask_size_{i}" for i in range(1, level+1)]].sum(axis=1)
    df[f"ofi_level_{level}"] = (bid_sum - ask_sum) / (bid_sum + ask_sum)

print(df[["ofi_top", "ofi_level_1", "ofi_level_3", "ofi_level_5", "ofi_weighted", "ofi_deep"]].describe())

             ofi_top    ofi_level_1    ofi_level_3    ofi_level_5  \
count  400391.000000  400391.000000  400391.000000  400391.000000   
mean        0.069742       0.069742       0.109846       0.094766   
std         0.569703       0.569703       0.440027       0.394565   
min        -0.999375      -0.999375      -0.989722      -0.975475   
25%        -0.333333      -0.333333      -0.191095      -0.173554   
50%         0.000000       0.000000       0.124304       0.102119   
75%         0.587302       0.587302       0.438330       0.381443   
max         0.999513       0.999513       0.994993       0.983145   

        ofi_weighted       ofi_deep  
count  400391.000000  400391.000000  
mean        0.085491       0.073022  
std         0.358546       0.347820  
min        -0.969092      -0.921767  
25%        -0.156468      -0.164200  
50%         0.094513       0.084043  
75%         0.340816       0.321769  
max         0.970927       0.955301  


In [26]:
# Microprice — weighted by opposite-side volume
df["microprice"] = (
    df["bid_price_1"] * df["ask_size_1"] + df["ask_price_1"] * df["bid_size_1"]
) / (df["bid_size_1"] + df["ask_size_1"])

# Micro minus mid — a normalized "tilt" signal
df["micro_minus_mid"] = df["microprice"] - df["mid_price"]

print(df[["mid_price", "microprice", "micro_minus_mid"]].head(10))
print()
print(df["micro_minus_mid"].describe())

   mid_price  microprice  micro_minus_mid
0    585.635  585.380367        -0.254633
1    585.635  585.380367        -0.254633
2    585.635  585.380367        -0.254633
3    585.620  585.620000         0.000000
4    585.620  585.620000         0.000000
5    585.620  585.620000         0.000000
6    585.620  585.620000         0.000000
7    585.620  585.620000         0.000000
8    585.620  585.620000         0.000000
9    585.620  585.620000         0.000000

count    400391.000000
mean          0.004537
std           0.049511
min          -0.409971
25%          -0.021667
50%           0.000000
75%           0.035000
max           0.391070
Name: micro_minus_mid, dtype: float64


In [27]:
# How many rows = ~1 second of trading? AAPL has ~400K events / 6.5 hours = ~17/sec
# So rolling windows of 50, 200, 1000 capture short/medium/long memory

# Rolling mean of OFI — sustained pressure
df["ofi_top_roll_50"] = df["ofi_top"].rolling(window=50, min_periods=1).mean()
df["ofi_top_roll_200"] = df["ofi_top"].rolling(window=200, min_periods=1).mean()

# Rolling volatility of mid-price (how jumpy is the price?)
df["mid_volatility_50"] = df["mid_price"].rolling(window=50, min_periods=1).std()
df["mid_volatility_200"] = df["mid_price"].rolling(window=200, min_periods=1).std()

# Rolling spread — is the market getting wider?
df["spread_roll_50"] = df["spread"].rolling(window=50, min_periods=1).mean()

# Mid-price momentum: change over last N events
df["mid_change_50"] = df["mid_price"] - df["mid_price"].shift(50)
df["mid_change_200"] = df["mid_price"] - df["mid_price"].shift(200)

print(df[[
    "ofi_top_roll_50", "ofi_top_roll_200", 
    "mid_volatility_50", "mid_volatility_200",
    "mid_change_50", "mid_change_200"
]].describe())

       ofi_top_roll_50  ofi_top_roll_200  mid_volatility_50  \
count    400391.000000     400391.000000      400390.000000   
mean          0.069746          0.069738           0.018620   
std           0.417740          0.311417           0.012862   
min          -0.996915         -0.927454           0.000000   
25%          -0.213548         -0.137086           0.009552   
50%           0.063059          0.075235           0.016142   
75%           0.365582          0.285950           0.024850   
max           0.993869          0.980392           0.175457   

       mid_volatility_200  mid_change_50  mid_change_200  
count       400390.000000  400341.000000   400191.000000  
mean             0.040739      -0.001011       -0.004067  
std              0.021291       0.054149        0.116773  
min              0.000000      -0.580000       -0.835000  
25%              0.026081      -0.035000       -0.080000  
50%              0.036619       0.000000       -0.005000  
75%              0.

In [28]:
# Distance from best to next level (in $)
df["ask_gap_1to2"] = df["ask_price_2"] - df["ask_price_1"]
df["bid_gap_1to2"] = df["bid_price_1"] - df["bid_price_2"]

# Distance from best to level 5
df["ask_gap_1to5"] = df["ask_price_5"] - df["ask_price_1"]
df["bid_gap_1to5"] = df["bid_price_1"] - df["bid_price_5"]

# Book imbalance over deeper levels (alternative imbalance metric)
df["volume_imbalance_5"] = (
    df[[f"bid_size_{i}" for i in range(1, 6)]].sum(axis=1)
    - df[[f"ask_size_{i}" for i in range(1, 6)]].sum(axis=1)
)

print(df[["ask_gap_1to2", "bid_gap_1to2", "ask_gap_1to5", "bid_gap_1to5"]].describe())

        ask_gap_1to2   bid_gap_1to2  ask_gap_1to5   bid_gap_1to5
count  400391.000000  400391.000000  400391.00000  400391.000000
mean        0.028964       0.034239       0.10328       0.120130
std         0.029597       0.035725       0.05877       0.077911
min         0.010000       0.010000       0.04000       0.040000
25%         0.010000       0.010000       0.06000       0.080000
50%         0.020000       0.020000       0.09000       0.110000
75%         0.040000       0.040000       0.13000       0.140000
max         0.520000       0.910000       1.01000       1.440000


In [29]:
# Load the message file
MESSAGE_FILE = "../data/AAPL_2012-06-21_34200000_57600000_message_10.csv"
message_columns = ["time_sec", "event_type", "order_id", "size", "price", "direction"]
messages = pd.read_csv(MESSAGE_FILE, header=None, names=message_columns)

print(f"Message file shape: {messages.shape}")
print(f"Event types present: {messages['event_type'].value_counts().to_dict()}")

Message file shape: (400391, 6)
Event types present: {1: 191015, 3: 171126, 4: 23658, 5: 11332, 2: 3260}


In [30]:
# Indicator: was this event a new limit order? cancel? execution?
messages["is_new_order"] = (messages["event_type"] == 1).astype(int)
messages["is_cancel"] = (messages["event_type"].isin([2, 3])).astype(int)  # 2=partial cancel, 3=full delete
messages["is_execution"] = (messages["event_type"].isin([4, 5])).astype(int)  # 4=visible, 5=hidden

# Buyer vs seller initiated
messages["is_buy_event"] = (messages["direction"] == 1).astype(int)
messages["is_sell_event"] = (messages["direction"] == -1).astype(int)

In [31]:
# How many of each event type in last 50 messages?
messages["new_orders_last_50"] = messages["is_new_order"].rolling(50, min_periods=1).sum()
messages["cancels_last_50"] = messages["is_cancel"].rolling(50, min_periods=1).sum()
messages["executions_last_50"] = messages["is_execution"].rolling(50, min_periods=1).sum()
messages["buy_events_last_50"] = messages["is_buy_event"].rolling(50, min_periods=1).sum()
messages["sell_events_last_50"] = messages["is_sell_event"].rolling(50, min_periods=1).sum()

# A nice derived feature: buy/sell event ratio
messages["buy_sell_ratio_50"] = (messages["buy_events_last_50"] + 1) / (messages["sell_events_last_50"] + 1)
# The +1 prevents division by zero

In [32]:
# Merge message features into df (same row order, so just paste them in)
flow_features = [
    "new_orders_last_50", "cancels_last_50", "executions_last_50",
    "buy_events_last_50", "sell_events_last_50", "buy_sell_ratio_50"
]

for feat in flow_features:
    df[feat] = messages[feat].values

print(df[flow_features].describe())

       new_orders_last_50  cancels_last_50  executions_last_50  \
count       400391.000000    400391.000000       400391.000000   
mean            23.852699        21.776134            4.368108   
std              2.689303         3.514043            4.380898   
min              0.000000         0.000000            0.000000   
25%             22.000000        20.000000            1.000000   
50%             24.000000        22.000000            3.000000   
75%             26.000000        24.000000            7.000000   
max             48.000000        49.000000           48.000000   

       buy_events_last_50  sell_events_last_50  buy_sell_ratio_50  
count       400391.000000        400391.000000      400391.000000  
mean            21.924446            28.072494           1.291457  
std              8.639635             8.640914           3.644049  
min              0.000000             0.000000           0.019608  
25%             16.000000            23.000000           0.485714

In [33]:
# Time between consecutive events (in seconds)
df["time_delta"] = df["time_sec"].diff()
df["time_delta"] = df["time_delta"].fillna(0)  # first row has no previous, fill with 0

# Rate of events: how many events per second over last 50 events
# = 50 events / total time elapsed in last 50 events
df["event_rate_50"] = 50 / df["time_delta"].rolling(50, min_periods=1).sum().replace(0, 1e-9)

print(df[["time_delta", "event_rate_50"]].describe())

          time_delta  event_rate_50
count  400391.000000   4.003910e+05
mean        0.058443   1.250275e+05
std         0.208833   7.901833e+07
min         0.000000   1.321187e+00
25%         0.000073   1.237444e+01
50%         0.000639   2.521856e+01
75%         0.009320   6.267716e+01
max         8.155205   5.000000e+10


In [34]:
# List all feature columns (excluding raw price/size and timestamps)
feature_cols = [c for c in df.columns if c not in [
    "time_sec", "time_hour"
] and not c.startswith("ask_price_") and not c.startswith("bid_price_")
   and not c.startswith("ask_size_") and not c.startswith("bid_size_")]

print(f"Total engineered features: {len(feature_cols)}")
print()
for c in feature_cols:
    print(f"  - {c}")

Total engineered features: 36

  - mid_price
  - spread
  - ofi_top
  - ofi_deep
  - total_bid_size
  - total_ask_size
  - log_spread
  - bid_volume_top
  - ask_volume_top
  - total_volume_top
  - ofi_weighted
  - ofi_level_1
  - ofi_level_3
  - ofi_level_5
  - microprice
  - micro_minus_mid
  - ofi_top_roll_50
  - ofi_top_roll_200
  - mid_volatility_50
  - mid_volatility_200
  - spread_roll_50
  - mid_change_50
  - mid_change_200
  - ask_gap_1to2
  - bid_gap_1to2
  - ask_gap_1to5
  - bid_gap_1to5
  - volume_imbalance_5
  - new_orders_last_50
  - cancels_last_50
  - executions_last_50
  - buy_events_last_50
  - sell_events_last_50
  - buy_sell_ratio_50
  - time_delta
  - event_rate_50


In [35]:
print("=" * 60)
print("STAGE 3 DIAGNOSTIC")
print("=" * 60)

# 1. NaN check
nan_counts = df.isnull().sum()
nan_cols = nan_counts[nan_counts > 0]
print(f"\n1. Columns with NaN values: {len(nan_cols)}")
if len(nan_cols) > 0:
    print(nan_cols)
else:
    print("   ✅ No NaNs")

# 2. Shape check
print(f"\n2. DataFrame shape: {df.shape}")
print(f"   Expected rows: ~400,000")

# 3. Sanity check on key features
print("\n3. Key feature sanity checks:")
print(f"   mid_price range: ${df['mid_price'].min():.2f} to ${df['mid_price'].max():.2f}")
print(f"   ofi_top mean: {df['ofi_top'].mean():.4f} (should be near 0)")
print(f"   microprice ≈ mid_price: {np.allclose(df['microprice'], df['mid_price'], atol=0.5)} (should be True)")

# 4. Rolling features should have variation
print("\n4. Rolling feature variation (std):")
print(f"   ofi_top_roll_50 std: {df['ofi_top_roll_50'].std():.4f} (should be > 0)")
print(f"   mid_volatility_50 std: {df['mid_volatility_50'].std():.4f} (should be > 0)")
print(f"   event_rate_50 std: {df['event_rate_50'].std():.4f} (should be > 0)")

# 5. Message-flow features alignment
print("\n5. Message-flow feature ranges:")
print(f"   new_orders_last_50 max: {df['new_orders_last_50'].max():.0f} (should be ≤ 50)")
print(f"   executions_last_50 max: {df['executions_last_50'].max():.0f} (should be ≤ 50)")
print(f"   buy_sell_ratio_50 median: {df['buy_sell_ratio_50'].median():.2f} (should be near 1)")

print("\n" + "=" * 60)

STAGE 3 DIAGNOSTIC

1. Columns with NaN values: 4
mid_volatility_50       1
mid_volatility_200      1
mid_change_50          50
mid_change_200        200
dtype: int64

2. DataFrame shape: (400391, 78)
   Expected rows: ~400,000

3. Key feature sanity checks:
   mid_price range: $577.48 to $588.18
   ofi_top mean: 0.0697 (should be near 0)
   microprice ≈ mid_price: True (should be True)

4. Rolling feature variation (std):
   ofi_top_roll_50 std: 0.4177 (should be > 0)
   mid_volatility_50 std: 0.0129 (should be > 0)
   event_rate_50 std: 79018330.5997 (should be > 0)

5. Message-flow feature ranges:
   new_orders_last_50 max: 48 (should be ≤ 50)
   executions_last_50 max: 48 (should be ≤ 50)
   buy_sell_ratio_50 median: 0.79 (should be near 1)



In [36]:
df["event_rate_50"] = 50 / df["time_delta"].rolling(50, min_periods=1).sum().replace(0, 1e-9)

In [37]:
# Diagnose first — how many rows are problem rows?
print("Diagnosing event_rate_50:")
print(f"  Max value: {df['event_rate_50'].max():,.0f}")
print(f"  Median: {df['event_rate_50'].median():.2f}")
print(f"  99th percentile: {df['event_rate_50'].quantile(0.99):.2f}")
print(f"  Rows above 10000 events/sec: {(df['event_rate_50'] > 10000).sum()}")

Diagnosing event_rate_50:
  Max value: 50,000,000,000
  Median: 25.22
  99th percentile: 2460.07
  Rows above 10000 events/sec: 304


In [38]:
# Cap event_rate_50 at 1000 events/sec (anything higher is a clock-resolution artifact)
df["event_rate_50"] = df["event_rate_50"].clip(upper=1000)

# Recheck
print(f"\nAfter clipping:")
print(f"  Max value: {df['event_rate_50'].max():.2f}")
print(f"  Median: {df['event_rate_50'].median():.2f}")
print(f"  Std: {df['event_rate_50'].std():.2f}")


After clipping:
  Max value: 1000.00
  Median: 25.22
  Std: 196.49


In [40]:
df.to_parquet("/Users/anirudhgoyal/Desktop/Projects/lob-midprice-prediction/data/aapl_stage3_features.parquet")
print(f"Saved cleaned data: {df.shape}")

Saved cleaned data: (400391, 78)
